In [ ]:
import yaml
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import FancyArrowPatch
from matplotlib.patches import Circle, Polygon
import math
import os
from ament_index_python.packages import get_package_share_directory

TITLE = "COURSE"

def get_yaml_file_path(package_name="mission_planner_2", yaml_filename="eyeball.yaml"):
    """Get the path to the YAML file in the ROS 2 package"""
    try:
        package_share_directory = get_package_share_directory(package_name)
        yaml_file_path = os.path.join(package_share_directory, "cfg", yaml_filename)
        return yaml_file_path
    except Exception as e:
        print(f"Error finding package path: {e}")
        print(f"Make sure package '{package_name}' is built and sourced")
        return None

def load_coordinates(yaml_file=None, package_name="mission_planner_2", yaml_filename="eyeball.yaml"):
    """Load coordinates from YAML file"""
    if yaml_file is None:
        yaml_file = get_yaml_file_path(package_name, yaml_filename)
        if yaml_file is None:
            raise FileNotFoundError("Could not find YAML file in ROS 2 package")

    print(f"Loading coordinates from: {yaml_file}")
    with open(yaml_file, 'r') as file:
        data = yaml.safe_load(file)
    return data['map']

def create_octagon_obstacle(coordinates, octagon_radius=1.35):
    """
    Create octagon obstacle coordinates specifically for the "octagon" point

    Parameters:
    coordinates: list of coordinate dictionaries from YAML
    octagon_radius: radius of the octagon (side to side distance / 2) in meters

    Returns:
    octagon dictionary or None if octagon point not found
    """
    octagon_point = None

    # Find the octagon coordinate
    for coord in coordinates:
        if coord['name'] == 'octagon':
            octagon_point = coord
            break

    if octagon_point is None:
        print("Warning: 'octagon' coordinate not found")
        return None

    # Octagon position in NED
    center_north = octagon_point['x']
    center_east = octagon_point['y']

    # Create octagon vertices
    octagon_vertices = []
    num_sides = 8

    for i in range(num_sides):
        angle = 2 * math.pi * i / num_sides + math.pi/8
        # Calculate vertex position
        vertex_north = center_north + octagon_radius * math.cos(angle)
        vertex_east = center_east + octagon_radius * math.sin(angle)
        octagon_vertices.append((vertex_north, vertex_east))

    return {
        'name': 'octagon_obstacle',
        'center': (center_north, center_east),
        'vertices': octagon_vertices,
        'radius': octagon_radius
    }

def create_slalom_gates(coordinates, gate_spacing=2.0, gate_width=3.0, layer_2_offset=0.0, layer_3_offset=0.0):
    """
    Create slalom gate coordinates specifically for the "slalom_start" point

    Parameters:
    coordinates: list of coordinate dictionaries from YAML
    gate_spacing: distance between gate centers along yaw direction (m)
    gate_width: width of each gate (m)

    Returns:
    list of gate dictionaries or empty list if slalom_start not found
    """
    slalom_start = None

    # Find the slalom_start coordinate
    for coord in coordinates:
        if coord['name'] == 'slalom':
            slalom_start = coord
            break

    if slalom_start is None:
        print("Warning: 'slalom' coordinate not found")
        return []

    yaw_rad = math.radians(slalom_start.get('yaw', 0.0))  # Default to 0 if not specified

    # Slalom start position in NED
    start_north = slalom_start['x']
    start_east = slalom_start['y']

    gates = []

    # Create 3 gates
    for i in range(3):
        # Move forward along yaw direction: 2m, 4m, 6m from start
        distance_forward = gate_spacing * i
        gate_center_north = start_north + distance_forward * math.cos(yaw_rad)
        gate_center_east = start_east + distance_forward * math.sin(yaw_rad)

        # Create perpendicular line (rotate yaw by 90 degrees)
        perp_yaw_rad = yaw_rad + math.pi/2
        half_width = gate_width / 2

        # Gate endpoints perpendicular to yaw
        left_north = gate_center_north + half_width * math.cos(perp_yaw_rad)
        left_east = gate_center_east + half_width * math.sin(perp_yaw_rad)

        right_north = gate_center_north - half_width * math.cos(perp_yaw_rad)
        right_east = gate_center_east - half_width * math.sin(perp_yaw_rad)

        gates.append({
            'name': f'slalom_gate_{i+1}',
            'center': (gate_center_north, gate_center_east),
            'left_end': (left_north, left_east),
            'right_end': (right_north, right_east),
            'gate_number': i + 1
        })

    for key in ('center','left_end','right_end'):
        gates[1][key] = (gates[1][key][0], gates[1][key][1] + layer_2_offset)
        gates[2][key] = (gates[2][key][0], gates[2][key][1] + layer_3_offset)

    return gates

def create_gate_end_obstacle(coordinates, obstacle_setback=2.0, obstacle_half_width=1.5):
    """
    Create obstacle coordinate specifically for the "gate_end" point

    Parameters:
    coordinates: list of coordinate dictionaries from YAML
    obstacle_setback: distance to move back from gate_end along yaw direction (m)
    obstacle_half_width: half-width of obstacle extending perpendicular to yaw (m)

    Returns:
    obstacle dictionary or None if gate_end not found
    """
    gate_end = None

    # Find the gate_end coordinate
    for coord in coordinates:
        if coord['name'] == 'gate':
            gate_end = coord
            break

    if gate_end is None:
        print("Warning: 'gate_end' coordinate not found")
        return None

    yaw_rad = math.radians(gate_end.get('yaw', 0.0))  # Default to 0 if not specified

    # Gate position in NED
    gate_north = gate_end['x']
    gate_east = gate_end['y']

    # Move back 2m along the yaw direction
    # In NED: forward direction is yaw angle from North toward East
    setback_north = gate_north - obstacle_setback * math.cos(yaw_rad)
    setback_east = gate_east - obstacle_setback * math.sin(yaw_rad)

    # Create perpendicular line (rotate yaw by 90 degrees)
    perp_yaw_rad = yaw_rad + math.pi/2

    # Extend 1.5m each side perpendicular to yaw
    obstacle_start_north = setback_north + obstacle_half_width * math.cos(perp_yaw_rad)
    obstacle_start_east = setback_east + obstacle_half_width * math.sin(perp_yaw_rad)

    obstacle_end_north = setback_north - obstacle_half_width * math.cos(perp_yaw_rad)
    obstacle_end_east = setback_east - obstacle_half_width * math.sin(perp_yaw_rad)

    return {
        'name': 'gate_end_obstacle',
        'start': (obstacle_start_north, obstacle_start_east),
        'end': (obstacle_end_north, obstacle_end_east),
        'gate_name': 'gate_end'
    }

def plot_ned_coordinates_with_obstacles(coordinates, figsize=(12, 8), save_path=None,
                                      obstacle_setback=2.0, obstacle_half_width=1.5,
                                      gate_spacing=2.0, gate_width=3.0, octagon_radius=1.35, slalom_layer_2_offset=0.0, slalom_layer_3_offset=0.0):
    """
    Create a 2D visualization of robotics coordinates in NED frame with gate obstacles and octagon

    Parameters:
    coordinates: list of coordinate dictionaries from YAML
    figsize: tuple for figure size
    save_path: optional path to save the plot
    obstacle_setback: distance to move back from gate along yaw direction (m)
    obstacle_half_width: half-width of obstacle extending perpendicular to yaw (m)
    gate_spacing: distance between gate centers along yaw direction (m)
    gate_width: width of each gate (m)
    octagon_radius: radius of the octagon around "octagon" point (m)
    """

    fig, ax = plt.subplots(figsize=figsize)

    # Extract coordinates and metadata
    x_coords = []
    y_coords = []
    names = []
    yaws = []

    for coord in coordinates:
        x_coords.append(coord['x'])
        y_coords.append(coord['y'])
        names.append(coord['name'])
        yaws.append(coord.get('yaw', 0))  # Default yaw to 0 if not specified

    # Convert to numpy arrays for easier manipulation
    x_coords = np.array(x_coords)
    y_coords = np.array(y_coords)

    # Transform coordinates for NED visualization (swap x and y, then flip y)
    # In NED: X=North (up), Y=East (right)
    # For matplotlib: we plot Y (East) on x-axis and X (North) on y-axis
    plot_x = y_coords  # East coordinates go to matplotlib x-axis
    plot_y = x_coords  # North coordinates go to matplotlib y-axis

    # Plot points
    scatter = ax.scatter(plot_x, plot_y, c='red', s=50, alpha=0.7,
                        edgecolors='darkred', linewidth=2, zorder=5)

    # Add point labels
    for i, name in enumerate(names):
        ax.annotate(name, (plot_x[i], plot_y[i]),
                   xytext=(7, 7), textcoords='offset points',
                   fontsize=6, ha='left', va='bottom',
                   bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.7),
                   zorder=6)

    # Add orientation arrows for points with yaw
    arrow_length = 1.0
    for i, yaw in enumerate(yaws):
        # if yaw != 0:
        if True:
            # Convert yaw to radians (assuming yaw is in degrees)
            yaw_rad = math.radians(yaw)
            # In NED frame: yaw is measured from North (X) toward East (Y)
            # For display: North is up (matplotlib y), East is right (matplotlib x)
            dx_ned = arrow_length * math.sin(yaw_rad)  # East component (to matplotlib x)
            dy_ned = arrow_length * math.cos(yaw_rad)  # North component (to matplotlib y)

            arrow = FancyArrowPatch((plot_x[i], plot_y[i]),
                                  (plot_x[i] + dx_ned, plot_y[i] + dy_ned),
                                  arrowstyle='->', mutation_scale=20,
                                  color='blue', linewidth=2, zorder=4)
            ax.add_patch(arrow)

    # Create and plot torpedo visualization line
    torpedo_coord = None
    for coord in coordinates:
        if coord['name'] == 'torpedo_with_yaw':
            torpedo_coord = coord
            break

    if torpedo_coord:
        torpedo_yaw_rad = math.radians(torpedo_coord.get('yaw', 0))
        torpedo_north = torpedo_coord['x']
        torpedo_east = torpedo_coord['y']

        # 62cm = 0.62m line length, centered on torpedo coordinates
        line_half_length = 0.31  # 31cm each side

        # Calculate line endpoints along yaw direction
        start_north = torpedo_north - line_half_length * math.sin(torpedo_yaw_rad) #
        start_east = torpedo_east + line_half_length * math.cos(torpedo_yaw_rad)
        end_north = torpedo_north + line_half_length * math.sin(torpedo_yaw_rad)
        end_east = torpedo_east - line_half_length * math.cos(torpedo_yaw_rad)

        # Transform to plot coordinates (East to x, North to y)
        start_plot_x = start_north
        start_plot_y = start_east
        end_plot_x = end_north
        end_plot_y = end_east

        print(f"Torpedo start: ({start_plot_x}, {start_plot_y}), end: ({end_plot_x}, {end_plot_y})")

        # Plot torpedo visualization line
        ax.plot([start_plot_y, end_plot_y], [start_plot_x, end_plot_x],
               'orange', linewidth=4, alpha=0.9, solid_capstyle='round',
               label='Torpedo (60cm)', zorder=8)

    # Create and plot octagon obstacle
    octagon = create_octagon_obstacle(coordinates, octagon_radius)

    if octagon:
        # Transform vertices to plot coordinates (East to x, North to y)
        plot_vertices = []
        for vertex_north, vertex_east in octagon['vertices']:
            plot_vertices.append((vertex_east, vertex_north))  # (x, y) for matplotlib

        # Create and add octagon patch
        octagon_patch = Polygon(plot_vertices, closed=True,
                               facecolor='purple', alpha=0.6,
                               edgecolor='purple', linewidth=2,
                               label='Octagon Obstacle', zorder=6)
        ax.add_patch(octagon_patch)

        # Add octagon center point and label
        center_north, center_east = octagon['center']
        center_plot_x = center_east
        center_plot_y = center_north

    # Create and plot slalom gates
    slalom_gates = create_slalom_gates(coordinates, gate_spacing, gate_width, layer_2_offset=slalom_layer_2_offset, layer_3_offset=slalom_layer_3_offset)

    for gate in slalom_gates:
        center_north, center_east = gate['center']
        left_north, left_east = gate['left_end']
        right_north, right_east = gate['right_end']

        # Transform to plot coordinates (East to x, North to y)
        center_plot_x = center_east
        center_plot_y = center_north
        left_plot_x = left_east
        left_plot_y = left_north
        right_plot_x = right_east
        right_plot_y = right_north

        # Red center point
        ax.plot(center_plot_x, center_plot_y, 'ro', markersize=6, markeredgecolor='black',
               label='Slalom Gates' if gate == slalom_gates[0] else "", zorder=7)

        # White endpoint markers
        ax.plot([left_plot_x, right_plot_x], [left_plot_y, right_plot_y],
               'wo', markersize=6, markeredgecolor='black', markeredgewidth=1, zorder=7)

    # Create and plot obstacle for gate_end
    obstacle = create_gate_end_obstacle(coordinates, obstacle_setback, obstacle_half_width)

    if obstacle:
        start_north, start_east = obstacle['start']
        end_north, end_east = obstacle['end']

        # Transform to plot coordinates (East to x, North to y)
        start_plot_x = start_east
        start_plot_y = start_north
        end_plot_x = end_east
        end_plot_y = end_north

        # Plot obstacle as thick red line
        ax.plot([start_plot_x, end_plot_x], [start_plot_y, end_plot_y],
               'r-', linewidth=8, alpha=0.8, solid_capstyle='round',
               label='Gate End Obstacle', zorder=7)

    # Set up the plot with proper NED orientation
    ax.set_xlabel('East (Y) [m]', fontsize=12, fontweight='bold')
    ax.set_ylabel('North (X) [m]', fontsize=12, fontweight='bold')
    ax.set_title(TITLE, fontsize=14, fontweight='bold')

    # Set plot area to specified dimensions
    ax.set_xlim(-20, 20)  # East axis: -20m to +20m
    ax.set_ylim(-1, 20)   # North axis: -1m to +20m

    # Add custom grid at 2.8m intervals with 0.3m thick lines
    ax.grid(True, alpha=0.7, linestyle='-', linewidth=0.3*10, color='gray')  # Scale linewidth for visibility

    # Create grid ticks at 2.8m intervals centered on origin (0,0)
    grid_spacing = 2.8

    # Calculate grid lines that pass through (0,0)
    # For x-axis (East): find how many grid lines fit in each direction from 0
    x_min, x_max = -20, 20
    x_neg_count = int(np.ceil(abs(x_min) / grid_spacing))
    x_pos_count = int(np.ceil(abs(x_max) / grid_spacing))
    x_ticks = np.concatenate([
        np.arange(0, -x_neg_count * grid_spacing - grid_spacing/2, -grid_spacing)[::-1],
        np.arange(0, x_pos_count * grid_spacing + grid_spacing/2, grid_spacing)
    ])
    x_ticks = x_ticks[(x_ticks >= x_min) & (x_ticks <= x_max)]

    # For y-axis (North): find how many grid lines fit in each direction from 0
    y_min, y_max = -1, 20
    y_neg_count = int(np.ceil(abs(y_min) / grid_spacing))
    y_pos_count = int(np.ceil(abs(y_max) / grid_spacing))
    y_ticks = np.concatenate([
        np.arange(0, -y_neg_count * grid_spacing - grid_spacing/2, -grid_spacing)[::-1],
        np.arange(0, y_pos_count * grid_spacing + grid_spacing/2, grid_spacing)
    ])
    y_ticks = y_ticks[(y_ticks >= y_min) & (y_ticks <= y_max)]

    ax.set_xticks(x_ticks)
    ax.set_yticks(y_ticks)
    ax.set_aspect('equal')

    # Add origin marker
    ax.plot(0, 0, 'ko', markersize=8, label='Origin (0,0)', zorder=5)

    # Adjust layout
    plt.tight_layout()

    # Save if path provided
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Plot saved to {save_path}")

    plt.show()

    # Print coordinate summary
    print("\nCoordinate Summary (NED Frame):")
    print("-" * 50)
    print("Display: X-axis=East(Y), Y-axis=North(X)")
    for coord in coordinates:
        yaw_str = f", yaw: {coord['yaw']}°" if 'yaw' in coord else ""
        print(f"{coord['name']}: N={coord['x']:.1f}, E={coord['y']:.1f}")

    # Print octagon summary
    if octagon:
        print("\nOctagon Obstacle:")
        print("-" * 50)
        center_n, center_e = octagon['center']
        print(f"{octagon['name']}: Center(N={center_n:.1f}, E={center_e:.1f}), Radius={octagon['radius']:.1f}m")
        print("Vertices (NED):")
        for i, (vertex_n, vertex_e) in enumerate(octagon['vertices']):
            print(f"  Vertex {i+1}: N={vertex_n:.2f}, E={vertex_e:.2f}")

    # Print torpedo visualization summary
    if torpedo_coord:
        print("\nTorpedo Visualization:")
        print("-" * 50)
        torpedo_yaw_rad = math.radians(torpedo_coord.get('yaw', 0))
        torpedo_north = torpedo_coord['x']
        torpedo_east = torpedo_coord['y']
        line_half_length = 0.3

        start_north = torpedo_north - line_half_length * math.sin(torpedo_yaw_rad)
        start_east = torpedo_east - line_half_length * math.cos(torpedo_yaw_rad)
        end_north = torpedo_north + line_half_length * math.sin(torpedo_yaw_rad)
        end_east = torpedo_east + line_half_length * math.cos(torpedo_yaw_rad)

        print(f"Torpedo center: N={torpedo_north:.1f}, E={torpedo_east:.1f}, Yaw={torpedo_coord.get('yaw', 0):.1f}°")
        print(f"60cm line: Start(N={start_north:.2f}, E={start_east:.2f}) -> End(N={end_north:.2f}, E={end_east:.2f})")

    # Print slalom gates summary
    if slalom_gates:
        print("\nSlalom Gates:")
        print("-" * 50)
        for gate in slalom_gates:
            center_n, center_e = gate['center']
            left_n, left_e = gate['left_end']
            right_n, right_e = gate['right_end']
            print(f"{gate['name']}: Center(N={center_n:.1f}, E={center_e:.1f})")
            print(f"  Left(N={left_n:.1f}, E={left_e:.1f}) -> Right(N={right_n:.1f}, E={right_e:.1f})")

    # Print obstacle summary
    if obstacle:
        print("\nGate End Obstacle:")
        print("-" * 50)
        start_n, start_e = obstacle['start']
        end_n, end_e = obstacle['end']
        print(f"{obstacle['name']}: Start(N={start_n:.1f}, E={start_e:.1f}) -> End(N={end_n:.1f}, E={end_e:.1f})")

# Convenience function to use the original function name
def plot_ned_coordinates(coordinates, figsize=(12, 8), save_path=None):
    """Wrapper for backward compatibility"""
    return plot_ned_coordinates_with_obstacles(coordinates, figsize, save_path)

# Example usage
if __name__ == "__main__":
    try:
        # Load coordinates from ROS 2 package
        coords = load_coordinates()  # Uses mission_planner_2/cfg/static_tfs.yaml

        # Create the visualization with obstacles, slalom gates, and octagon
        plot_ned_coordinates_with_obstacles(coords,
                                          figsize=(12, 8),
                                          save_path='robotics_course_ned_with_obstacles.png',
                                          obstacle_setback=0.0,
                                          obstacle_half_width=1.5,
                                          gate_spacing=2.0,
                                          gate_width=3.0,
                                          octagon_radius=1.35,
                                          slalom_layer_2_offset = -0.2,
                                          slalom_layer_3_offset = -0.2,
                                          )

    except Exception as e:
        print(f"Error: {e}")
        print("Make sure your ROS 2 workspace is sourced and package is built.")

In [ ]:
import math

def torp_ang_aron(torp_x, torp_y, pool_edge_x, pool_edge_y):
    return 90+math.degrees(math.atan2( torp_y - pool_edge_y, torp_x - pool_edge_x))


In [ ]:
x1, y1 = 0, 0
x2, y2 = 1, 1

torp_ang_aron(x1, y1, x2, y2)

In [ ]:
x1, y1 = 1, 1
x2, y2 = 0, 0

torp_ang_aron(x1, y1, x2, y2)